## 6. Generate the trees.

In [ ]:
!~/miniconda3/bin/conda run -n nextstrain augur filter \
    --metadata 

The comand block below was taken from the zika tutoral: 
https://github.com/nextstrain/zika-tutorial/blob/main/Snakefile

It runs 11 augur comands that will:
- curate metadata
- align the sequences
- infer a ML three
- perform ancestral recosntruction on the tree
- calculate the traits on the tree
- trasnlate the alignment into protein
- infer the frequencies for the clades/traits over the time
- export the auspice json to be visualized.

some changes must be made on it before running such as: 
- **path for the files**
- **coluns and branch-labels** at augur traits

In [ ]:
### 229E

In [ ]:
#!~/miniconda3/bin/conda env list

# variables
WORKDIR="/Users/dt14/Desktop/HCoV/REFs/"

#/Users/dt14/Desktop/HCoV/REFs/229E/NCBI_virus/229E_sequences_09Sep26_formated.fasta
SEQ="$WORKDIR/dataset/rvi_gisaid_ncbi_rsv_a_seqs_n36888+1406.fasta"
META="$WORKDIR/dataset/rsva_ids_metadata.tsv"
REFSEQ="$WORKDIR/dataset/RSVA_MZ515597.gb"
CLADES="$WORKDIR/dataset/lineages.tsv"
COLORS="$WORKDIR/colors.tsv"
CONFIG="$WORKDIR/auspice_config.json"
RESULTS="$WORKDIR/results"
TREADS=4

# code
echo "1 - Running augur index"
augur index \
        --sequences $SEQ \
        --output $RESULTS/sequence_index.tsv

echo "2 - Running augur filter"
augur filter \
        --sequences $SEQ \
        --sequence-index $RESULTS/sequence_index.tsv \
        --metadata $META \
        --output-sequences $RESULTS/filtered_sequences.fasta \
        --output-metadata $RESULTS/filtered_metadata.tsv

echo "3 - Running augur align"
augur align \
        --sequences $RESULTS/filtered_sequences.fasta \
        --reference-sequence $REFSEQ \
        --output $RESULTS/aligment.fasta \
        --fill-gaps \
        --remove-reference \
        --nthreads $TREADS

echo "4 - Running augur tree"

#cmaple -aln $RESULTS/aligment.fasta --sprta -nt $TREADS

augur tree \
        --alignment $RESULTS/aligment.fasta \
        --nthreads $TREADS \
        --output $RESULTS/aligment.fasta.treefile

echo "5 - Running augur refine"
augur refine \
        --tree $RESULTS/aligment.fasta.treefile \
        --alignment $RESULTS/aligment.fasta \
        --metadata $RESULTS/filtered_metadata.tsv \
        --output-tree $RESULTS/time_solved.tree \
        --output-node-data $RESULTS/time_solved_node_data.json \
        --timetree \
        --coalescent 'opt' \
        --date-confidence \
        --date-inference 'joint' \
        --root 'best' \
        --seed 1234567

echo "6 - Running augur ancestral"
augur ancestral \
        --tree $RESULTS/time_solved.tree \
        --alignment $RESULTS/aligment.fasta \
        --root-sequence $REFSEQ \
        --output-node-data $RESULTS/ancestral.json

echo "7 - Running augur traits"
augur traits \
        --tree $RESULTS/time_solved.tree \
        --metadata $RESULTS/filtered_metadata.tsv \
        --output-node-data $RESULTS/traits.json \
        --columns 'country' 'country2' 'clade' \
        --branch-labels 'country' 'country2' 'clade' \
        --confidence \
        --branch-confidence country=70 clade=70 country2=70

echo "8 - Running augur translate"
augur translate \
        --tree $RESULTS/time_solved.tree \
        --ancestral-sequences $RESULTS/ancestral.json \
        --reference-sequence  $REFSEQ \
        --output-node-data $RESULTS/translate.json \
        --alignment-output $RESULTS/my_alignment_%GENE.fasta

#echo "9 - Running augur clades"
#augur clades \
#        --tree $RESULTS/time_solved.tree \
#        --mutations $RESULTS/traits.json \
#        --clades $CLADES \
#        --output-node-data $RESULTS/clades.json

echo "10 - Running augur frquences"
augur frequencies \
        --metadata $RESULTS/filtered_metadata.tsv \
        --tree $RESULTS/time_solved.tree \
        #--min-date '2021-06-01' \
        --method kde \
        --pivot-interval 1 \
        --include-internal-nodes \
        --pivot-interval-units weeks \
        --output $RESULTS/auspice_output_tip-frequencies.json

# The name of the output file to generate the frequences should be the output from
# augur export v2 + "_tip-frequencies"
echo "11 - Running augur export"
augur export v2 \
        --tree $RESULTS/time_solved.tree \
        --metadata $RESULTS/filtered_metadata.tsv \
        --node-data $RESULTS/time_solved_node_data.json $RESULTS/traits.json $RESULTS/ancestral.json $RESULTS/translate.json \
        --colors $COLORS \
        --auspice-config $CONFIG \
        --description "" \
        --output $RESULTS/auspice_output.json